# Q24: Types of Model Drift
**Topic:** MLOps | **Difficulty:** Medium | **Time:** 15 min
**Sheet:** Grind75ML

---

## Question
What types of Model Drifts have you encountered, and how have you addressed them?

## Detailed Answer

### 1. Data Drift (Covariate Shift)
**What:** The distribution of input features changes.
$$P_{train}(X) \neq P_{production}(X)$$
**Example:** A model trained on summer data deployed in winter — weather features change.
**Detection:** Statistical tests (KS test, PSI) on feature distributions.
**Fix:** Retrain on recent data, use domain adaptation.

### 2. Concept Drift
**What:** The relationship between features and target changes.
$$P_{train}(Y|X) \neq P_{production}(Y|X)$$
**Example:** COVID changed the relationship between economic indicators and consumer behavior.
**Types:** Sudden, gradual, incremental, recurring.
**Detection:** Monitor model performance metrics over time.
**Fix:** Retrain, online learning, ensemble of models from different time windows.

### 3. Label Drift (Prior Probability Shift)
**What:** The distribution of the target variable changes.
$$P_{train}(Y) \neq P_{production}(Y)$$
**Example:** Fraud rate changes from 1% to 5% — class imbalance shifts.
**Detection:** Monitor label distribution in production feedback loop.
**Fix:** Adjust decision thresholds, retrain with balanced sampling.

### 4. Feature Drift (Schema Drift)
**What:** Features become unavailable, change format, or new features arrive.
**Example:** A data provider changes their API, a feature column becomes NULL.
**Detection:** Schema validation, null rate monitoring.
**Fix:** Feature contracts, fallback values, robust feature pipelines.

### Detection Framework
| Method | Measures | Tool |
|--------|----------|------|
| PSI (Population Stability Index) | Feature distribution shift | EvidentlyAI |
| KS Test | Distribution difference | scipy |
| Performance monitoring | Accuracy/F1 over time | MLflow, Grafana |
| Adversarial validation | Train vs prod distinguishability | Custom |

### Mitigation Strategies
1. **Scheduled retraining**: Retrain weekly/monthly on recent data
2. **Online learning**: Continuously update model with new data
3. **Champion/Challenger**: Always have a candidate model being tested
4. **Monitoring dashboards**: EvidentlyAI, Whylabs, Arize
5. **Automated rollback**: Revert to previous model if performance drops

In [ ]:
import numpy as np
from scipy import stats


def psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
    """Population Stability Index: measures distribution shift.
    PSI < 0.1: no shift, 0.1-0.25: moderate shift, > 0.25: significant shift.
    """
    # Bin the expected distribution
    breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
    breakpoints[0], breakpoints[-1] = -np.inf, np.inf
    
    expected_counts = np.histogram(expected, breakpoints)[0] / len(expected)
    actual_counts = np.histogram(actual, breakpoints)[0] / len(actual)
    
    # Avoid log(0)
    expected_counts = np.clip(expected_counts, 1e-6, None)
    actual_counts = np.clip(actual_counts, 1e-6, None)
    
    return np.sum((actual_counts - expected_counts) * np.log(actual_counts / expected_counts))


# Simulate drift
np.random.seed(42)
train_feature = np.random.normal(0, 1, 1000)
prod_no_drift = np.random.normal(0, 1, 1000)
prod_mild_drift = np.random.normal(0.5, 1.2, 1000)
prod_severe_drift = np.random.normal(2, 2, 1000)

for name, prod in [('No drift', prod_no_drift), ('Mild drift', prod_mild_drift), ('Severe drift', prod_severe_drift)]:
    psi_val = psi(train_feature, prod)
    ks_stat, ks_p = stats.ks_2samp(train_feature, prod)
    print(f'{name:<15} PSI={psi_val:.4f}  KS_stat={ks_stat:.4f}  KS_p={ks_p:.6f}')

## Key Takeaways
- **Data drift** is most common and easiest to detect; **concept drift** is hardest and most impactful
- Use **PSI > 0.25** or **KS test p < 0.05** as drift alerts
- **Performance monitoring** is the ultimate drift detector — if accuracy drops, something drifted
- Every production ML system needs a **monitoring + retraining pipeline**